# Group 2: Markowitz Optimisation with the XRD Covariance Matrix

Use the XRD covariance matrix for risk, build the efficient frontier, and pick the
**maximum Sharpe ratio** portfolio with a risk-free rate of 0.

**Stage 1:** the only constraint is a minimum holding of 0% for every stock (long-only, fully
invested).

**Later stages (only if needed):** look at the max-SR portfolio from the previous stage and
- cap every holding **above 10%** at a maximum of 10%,
- give every holding **below 0.1%** (including 0%) a minimum of 0.1%,
- leave holdings between 0.1% and 10% with no constraint other than >= 0.

Constraints from earlier stages are kept, and new ones are added for any new violations.
Repeat until every holding is between 0.1% and 10%. The max-SR portfolio and the efficient
frontier of every stage go into one Excel workbook.

**Expected returns:** the mean of each stock's stagger 2 time-weighted returns. The XRD
matrix only gives risk, so this is the return input; change `expected_returns` below if the
class uses a different one.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linprog, minimize

DATA_FILE = Path("../data/companies_and_staggered_returns.xlsx")
STAGGER = "stagger_2"

# The XRD covariance matrix from the professor. Put the file in this folder and set the name
# here. Until it arrives, the notebook falls back to our own matrix so it can be tested.
XRD_FILE = Path("XRD_covariance.xlsx")
XRD_SHEET = 0                    # sheet name or position in the XRD workbook
XRD_IS_ANNUALISED = False        # True if the XRD matrix is in annual units
OWN_COV_FILE = Path("group2_return_covariance.xlsx")

RISK_FREE_RATE = 0.0             # per period
MIN_HOLDING = 0.001              # 0.1%
MAX_HOLDING = 0.10               # 10%
FRONTIER_POINTS = 40
TOL = 1e-6                       # numerical tolerance when checking the 0.1% / 10% limits

OUTPUT_FILE = Path("group2_markowitz.xlsx")

## Stocks and expected returns

In [ ]:
companies = pd.read_excel(DATA_FILE, sheet_name="Companies")
stocks = (
    companies
    .dropna(subset=["sector"])
    .drop_duplicates(subset=["ticker"])
    .reset_index(drop=True)
)
tickers = stocks["ticker"].tolist()

returns = pd.read_excel(
    DATA_FILE, sheet_name=STAGGER, skiprows=254, nrows=80, parse_dates=["date"]
).set_index("date")
stock_returns = returns[tickers].astype(float)

# Return periods per year, from the spacing of the dates (28 days -> about 13.04)
period_days = stock_returns.index.to_series().diff().dt.days.median()
PERIODS_PER_YEAR = 365.25 / period_days

# Expected return per period for each stock
expected_returns = stock_returns.mean()

print(f"{len(tickers)} stocks, {len(stock_returns)} periods of {period_days:.0f} days "
      f"({PERIODS_PER_YEAR:.2f} per year)")
print(f"Expected return per period: {expected_returns.min():.4f} to {expected_returns.max():.4f}")

## The XRD covariance matrix

The matrix is read with tickers down the first column and across the top row. If it covers
more stocks than ours (for example the other groups' stocks too), only our 43 are used.

In [ ]:
def load_covariance(path, sheet, tickers):
    raw = pd.read_excel(path, sheet_name=sheet, index_col=0)
    raw.index = raw.index.astype(str).str.strip()
    raw.columns = raw.columns.astype(str).str.strip()
    missing = [t for t in tickers if t not in raw.index or t not in raw.columns]
    if missing:
        raise ValueError(f"these tickers are not in {path.name}: {missing}")
    extra = len(raw.index) - len(tickers)
    if extra > 0:
        print(f"{path.name} has {extra} more stocks than ours; using our {len(tickers)}.")
    return raw.loc[tickers, tickers].astype(float)


if XRD_FILE.exists():
    covariance = load_covariance(XRD_FILE, XRD_SHEET, tickers)
    if XRD_IS_ANNUALISED:
        covariance = covariance / PERIODS_PER_YEAR   # match the per-period expected returns
    COV_SOURCE = f"XRD covariance matrix ({XRD_FILE.name})"
else:
    covariance = load_covariance(OWN_COV_FILE, "Covariance", tickers)
    COV_SOURCE = f"OWN stagger 2 matrix ({OWN_COV_FILE.name}) - XRD file not found"
    print("*** XRD file not found: using our own covariance matrix for now. ***")

# Checks
asym = np.abs(covariance.values - covariance.values.T).max()
assert asym < 1e-10, f"covariance matrix is not symmetric (max difference {asym:.2e})"
covariance = (covariance + covariance.T) / 2
min_eig = np.linalg.eigvalsh(covariance.values).min()
assert min_eig > 0, f"covariance matrix is not positive definite (smallest eigenvalue {min_eig:.2e})"

# Units check: the XRD variances should be on the same scale as our own per-period variances
ratio = (np.diag(covariance) / stock_returns.var()).median()
print(f"Using: {COV_SOURCE}")
print(f"Smallest eigenvalue {min_eig:.2e}; median variance vs our own: {ratio:.2f}x")
if not 0.3 < ratio < 3:
    print("*** The variances are on a very different scale from our per-period returns. "
          "Check XRD_IS_ANNUALISED. (The weights are unaffected, but the reported "
          "returns, volatilities and Sharpe ratios would be.) ***")

## The optimiser

All three problems are solved with SciPy:

- **Minimum variance** for a target return (or with no target, for the global minimum
  variance portfolio), fully invested, each weight within its bounds.
- **Maximum return** (a linear programme), which gives the top end of the frontier.
- **Maximum Sharpe ratio.** Maximising return / volatility directly is awkward, so it uses the
  standard trick: substitute y = k w (k > 0), minimise y'Σy subject to (μ - r_f)'y = 1 and
  lb·sum(y) <= y <= ub·sum(y), then set w = y / sum(y). That is a convex problem with one
  answer, and it handles the minimum and maximum holding limits exactly.

In [ ]:
def _solve(objective, gradient, x0, bounds, constraints):
    res = minimize(objective, x0, jac=gradient, bounds=bounds, constraints=constraints,
                   method="SLSQP", options={"ftol": 1e-12, "maxiter": 5000})
    # SLSQP sometimes reports "positive directional derivative" when it is already at the
    # optimum and cannot improve further; accept that only if every constraint is met.
    feasible = all(
        (np.abs(c["fun"](res.x)) < 1e-8).all() if c["type"] == "eq" else (c["fun"](res.x) > -1e-8).all()
        for c in constraints
    ) and all((lo is None or x >= lo - 1e-8) and (hi is None or x <= hi + 1e-8)
              for x, (lo, hi) in zip(res.x, bounds))
    if not (res.success or (res.status == 8 and feasible)):
        raise RuntimeError(f"optimiser failed: {res.message}")
    return res.x


def _tidy(w, lb, ub):
    w = np.clip(w, lb, ub)
    return w / w.sum()


def portfolio_stats(w, mu, cov, rf=RISK_FREE_RATE):
    ret = float(mu @ w)
    vol = float(np.sqrt(w @ cov @ w))
    return ret, vol, (ret - rf) / vol


def max_return(mu, lb, ub):
    res = linprog(-mu, A_eq=np.ones((1, len(mu))), b_eq=[1], bounds=list(zip(lb, ub)),
                  method="highs")
    if not res.success:
        raise RuntimeError(f"max-return LP failed: {res.message}")
    return res.x


def min_variance(cov, lb, ub, mu=None, target=None):
    n = len(lb)
    S = cov / np.mean(np.diag(cov))                      # rescale for the solver
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(n)}]
    if target is not None:
        scale = np.abs(mu).max()
        cons.append({"type": "eq", "fun": lambda w: (mu @ w - target) / scale,
                     "jac": lambda w: mu / scale})
    start = lb + (ub - lb) * (1 - lb.sum()) / (ub - lb).sum()   # a feasible start
    w = _solve(lambda w: w @ S @ w, lambda w: 2 * S @ w, start, list(zip(lb, ub)), cons)
    return _tidy(w, lb, ub)


def max_sharpe(mu, cov, lb, ub, rf=RISK_FREE_RATE):
    n = len(mu)
    excess = mu - rf
    w_top = max_return(excess, lb, ub)
    if excess @ w_top <= 0:
        raise ValueError("no allowed portfolio has a positive excess return, so there is "
                         "no maximum Sharpe ratio portfolio")
    S = cov / np.mean(np.diag(cov))
    m = excess / np.abs(excess).max()
    ones = np.ones(n)
    A_lo = np.eye(n) - np.outer(lb, ones)                # y_i - lb_i * sum(y) >= 0
    A_hi = np.outer(ub, ones) - np.eye(n)                # ub_i * sum(y) - y_i >= 0
    cons = [
        {"type": "eq", "fun": lambda y: m @ y - 1, "jac": lambda y: m},
        {"type": "ineq", "fun": lambda y: A_lo @ y, "jac": lambda y: A_lo},
        {"type": "ineq", "fun": lambda y: A_hi @ y, "jac": lambda y: A_hi},
    ]
    # start from a feasible mix of the equal-ish start and the max-return portfolio
    w0 = 0.5 * (lb + (ub - lb) * (1 - lb.sum()) / (ub - lb).sum()) + 0.5 * w_top
    if m @ w0 <= 0:
        w0 = w_top
    y = _solve(lambda y: y @ S @ y, lambda y: 2 * S @ y, w0 / (m @ w0),
               [(0, None)] * n, cons)
    return _tidy(y / y.sum(), lb, ub)


def efficient_frontier(mu, cov, lb, ub, points=FRONTIER_POINTS):
    """Frontier from the minimum-variance portfolio up to the maximum-return portfolio."""
    w_min = min_variance(cov, lb, ub)
    w_top = max_return(mu, lb, ub)
    targets = np.linspace(mu @ w_min, mu @ w_top, points)
    weights = [w_min]
    for t in targets[1:-1]:
        weights.append(min_variance(cov, lb, ub, mu=mu, target=t))
    weights.append(w_top)
    return weights

## Run the stages

Each stage solves for the max-SR portfolio and the frontier under the current constraints,
then checks the holdings. Any holding above 10% gets a 10% cap and any holding below 0.1%
gets a 0.1% floor, on top of the constraints already in place. It stops once every holding
is between 0.1% and 10%.

In [ ]:
mu = expected_returns.values
cov = covariance.values
n = len(tickers)

lower = np.zeros(n)          # stage 1: minimum holding 0% (long-only)
upper = np.ones(n)           # no maximum
stages = []

for stage in range(1, 2 * n + 2):    # each stage adds at least one new constraint
    w = max_sharpe(mu, cov, lower, upper)
    frontier = efficient_frontier(mu, cov, lower, upper)
    too_big = w > MAX_HOLDING + TOL
    too_small = w < MIN_HOLDING - TOL
    stages.append({
        "stage": stage,
        "weights": pd.Series(w, index=tickers),
        "lower": pd.Series(lower.copy(), index=tickers),
        "upper": pd.Series(upper.copy(), index=tickers),
        "frontier": frontier,
        "too_big": [t for t, b in zip(tickers, too_big) if b],
        "too_small": [t for t, s in zip(tickers, too_small) if s],
    })
    ret, vol, sr = portfolio_stats(w, mu, cov)
    print(f"Stage {stage}: SR {sr:.4f}, {int((w > TOL).sum())} holdings, "
          f"max {w.max():.2%}, min {w.min():.2%} -> "
          f"{too_big.sum()} above 10%, {too_small.sum()} below 0.1%")
    if not too_big.any() and not too_small.any():
        print("All holdings are between 0.1% and 10%. Done.")
        break
    # add constraints only for the holdings that broke a limit; keep the earlier ones
    upper[too_big] = MAX_HOLDING
    lower[too_small] = MIN_HOLDING
else:
    raise RuntimeError("did not converge")

## Results

In [ ]:
annual_ret = lambda r: r * PERIODS_PER_YEAR
annual_vol = lambda v: v * np.sqrt(PERIODS_PER_YEAR)

summary_rows = []
for s in stages:
    w = s["weights"].values
    ret, vol, sr = portfolio_stats(w, mu, cov)
    summary_rows.append({
        "Stage": s["stage"],
        "Max-10% caps in force": int((s["upper"] < 1).sum()),
        "Min-0.1% floors in force": int((s["lower"] > 0).sum()),
        "Expected return (per period)": ret,
        "Volatility (per period)": vol,
        "Sharpe ratio (per period)": sr,
        "Expected return (annual)": annual_ret(ret),
        "Volatility (annual)": annual_vol(vol),
        "Sharpe ratio (annual)": sr * np.sqrt(PERIODS_PER_YEAR),
        "Holdings > 0": int((w > TOL).sum()),
        "Largest holding": w.max(),
        "Smallest holding": w.min(),
        "Holdings above 10%": ", ".join(s["too_big"]) or "none",
        "Holdings below 0.1%": ", ".join(s["too_small"]) or "none",
    })
summary = pd.DataFrame(summary_rows)

max_sr = pd.DataFrame({f"Stage {s['stage']}": s["weights"] for s in stages})
final = stages[-1]["weights"]
assert abs(final.sum() - 1) < 1e-8
assert final.min() >= MIN_HOLDING - TOL and final.max() <= MAX_HOLDING + TOL

summary.T

In [ ]:
(max_sr * 100).round(2).assign(sector=stocks.set_index("ticker")["sector"])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5.5))
for s in stages:
    pts = np.array([portfolio_stats(w, mu, cov)[:2] for w in s["frontier"]])
    line, = ax.plot(annual_vol(pts[:, 1]), annual_ret(pts[:, 0]), label=f"Stage {s['stage']} frontier")
    ret, vol, sr = portfolio_stats(s["weights"].values, mu, cov)
    ax.scatter(annual_vol(vol), annual_ret(ret), color=line.get_color(), edgecolor="black",
               zorder=3, s=60, label=f"Stage {s['stage']} max SR ({sr * np.sqrt(PERIODS_PER_YEAR):.2f})")
ax.set_xlabel("Volatility (annual)")
ax.set_ylabel("Expected return (annual)")
ax.set_title("Efficient frontier and max-Sharpe portfolio at each stage")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(alpha=0.2)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Save to Excel

- **Summary**: one row per stage (return, volatility and Sharpe ratio, per period and
  annual; how many caps and floors were in force; which holdings broke a limit).
- **Max SR Portfolios**: the max-SR weights at every stage, with weights outside
  0.1%-10% highlighted.
- **Constraints**: the minimum and maximum holding for each stock at each stage.
- **Efficient Frontiers**: every stage's frontier side by side, with a chart.
- **Frontier S1, S2, ...**: the weights of every point on each stage's frontier.
- **Inputs** and **Notes**: the expected returns and volatilities used, and the settings.

In [ ]:
from openpyxl.chart import Reference, ScatterChart, Series
from openpyxl.styles import Font, PatternFill

PCT, NUM = "0.00%", "0.0000"
bold = Font(bold=True)
red = PatternFill("solid", fgColor="F4CCCC")
info = stocks.set_index("ticker")[["sector", "currency"]]


def autosize(ws, width=14, first=10):
    for i, col in enumerate(ws.iter_cols(min_row=1, max_row=1), start=1):
        ws.column_dimensions[col[0].column_letter].width = first if i == 1 else width


with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    # Summary
    summary.to_excel(writer, sheet_name="Summary", index=False)
    ws = writer.sheets["Summary"]
    for row in ws.iter_rows(min_row=2):
        for c in row:
            header = ws.cell(row=1, column=c.column).value
            if "return" in header or "Volatility" in header or "holding" in header.lower():
                c.number_format = PCT
            elif "Sharpe" in header:
                c.number_format = NUM
    autosize(ws, 16, 8)

    # Max SR portfolios (highlight anything outside 0.1% - 10%)
    out = info.join(max_sr)
    out.rename_axis("ticker").to_excel(writer, sheet_name="Max SR Portfolios")
    ws = writer.sheets["Max SR Portfolios"]
    for row in ws.iter_rows(min_row=2, min_col=4):
        for c in row:
            c.number_format = PCT
            if c.value is not None and not (MIN_HOLDING - TOL <= c.value <= MAX_HOLDING + TOL):
                c.fill = red
    total_row = ws.max_row + 1
    ws.cell(row=total_row, column=1, value="Total").font = bold
    for j in range(4, ws.max_column + 1):
        col = ws.cell(row=1, column=j).column_letter
        c = ws.cell(row=total_row, column=j, value=f"=SUM({col}2:{col}{total_row - 1})")
        c.number_format, c.font = PCT, bold
    ws.freeze_panes = "D2"
    autosize(ws, 11)
    ws.column_dimensions["B"].width = 24

    # Constraints
    cons = info.copy()
    for s in stages:
        cons[f"Stage {s['stage']} min"] = s["lower"]
        cons[f"Stage {s['stage']} max"] = s["upper"].where(s["upper"] < 1)   # blank = no cap
    cons.rename_axis("ticker").to_excel(writer, sheet_name="Constraints")
    ws = writer.sheets["Constraints"]
    for row in ws.iter_rows(min_row=2, min_col=4):
        for c in row:
            c.number_format = PCT
    ws.freeze_panes = "D2"
    autosize(ws, 12)
    ws.column_dimensions["B"].width = 24

    # Efficient frontiers side by side, plus the max SR point of each stage
    blocks = []
    for s in stages:
        pts = [portfolio_stats(w, mu, cov) for w in s["frontier"]]
        blocks.append(pd.DataFrame({
            f"S{s['stage']} return (annual)": [annual_ret(p[0]) for p in pts],
            f"S{s['stage']} volatility (annual)": [annual_vol(p[1]) for p in pts],
            f"S{s['stage']} Sharpe (annual)": [p[2] * np.sqrt(PERIODS_PER_YEAR) for p in pts],
        }))
    frontiers = pd.concat(blocks, axis=1)
    frontiers.insert(0, "Point", range(1, len(frontiers) + 1))
    frontiers.to_excel(writer, sheet_name="Efficient Frontiers", index=False)
    ws = writer.sheets["Efficient Frontiers"]
    n_pts = len(frontiers)
    msr_row = n_pts + 3
    ws.cell(row=msr_row, column=1, value="Max SR").font = bold
    for k, s in enumerate(stages):
        ret, vol, sr = portfolio_stats(s["weights"].values, mu, cov)
        for j, v in enumerate([annual_ret(ret), annual_vol(vol), sr * np.sqrt(PERIODS_PER_YEAR)]):
            ws.cell(row=msr_row, column=2 + 3 * k + j, value=v).font = bold
    for row in ws.iter_rows(min_row=2, min_col=2):
        for c in row:
            header = ws.cell(row=1, column=c.column).value or ""
            c.number_format = NUM if "Sharpe" in header else PCT
    autosize(ws, 15, 8)

    chart = ScatterChart()
    chart.title = "Efficient frontier and max SR portfolio by stage"
    chart.style = 13
    chart.x_axis.title = "Volatility (annual)"
    chart.y_axis.title = "Expected return (annual)"
    chart.x_axis.number_format = chart.y_axis.number_format = "0%"
    chart.x_axis.delete = chart.y_axis.delete = False
    chart.height, chart.width = 11, 20
    chart.scatterStyle = "lineMarker"
    all_vols = frontiers[[c for c in frontiers.columns if "volatility" in c]].values
    all_rets = frontiers[[c for c in frontiers.columns if "return" in c]].values
    chart.x_axis.scaling.min = np.floor(np.nanmin(all_vols) * 50) / 50
    chart.y_axis.scaling.min = np.floor(np.nanmin(all_rets) * 50) / 50
    colours = ["1F77B4", "D62728", "2CA02C", "FF7F0E", "9467BD", "8C564B", "E377C2", "17BECF"]
    for k, s in enumerate(stages):
        colour = colours[k % len(colours)]
        ret_col, vol_col = 2 + 3 * k, 3 + 3 * k
        line = Series(Reference(ws, min_col=ret_col, min_row=2, max_row=n_pts + 1),
                      Reference(ws, min_col=vol_col, min_row=2, max_row=n_pts + 1),
                      title=f"Stage {s['stage']} frontier")
        line.marker.symbol = "none"
        line.smooth = False
        line.graphicalProperties.line.solidFill = colour
        line.graphicalProperties.line.width = 22000
        chart.series.append(line)
        point = Series(Reference(ws, min_col=ret_col, min_row=msr_row),
                       Reference(ws, min_col=vol_col, min_row=msr_row),
                       title=f"Stage {s['stage']} max SR")
        point.marker.symbol = "diamond"
        point.marker.size = 10
        point.marker.graphicalProperties.solidFill = colour
        point.marker.graphicalProperties.line.solidFill = "000000"
        point.graphicalProperties.line.noFill = True
        chart.series.append(point)
    ws.add_chart(chart, ws.cell(row=msr_row + 3, column=2).coordinate)

    # Weights of every frontier point, one sheet per stage
    for s in stages:
        fw = pd.DataFrame(np.array(s["frontier"]), columns=tickers)
        pts = [portfolio_stats(w, mu, cov) for w in s["frontier"]]
        fw.insert(0, "Sharpe (annual)", [p[2] * np.sqrt(PERIODS_PER_YEAR) for p in pts])
        fw.insert(0, "Volatility (annual)", [annual_vol(p[1]) for p in pts])
        fw.insert(0, "Return (annual)", [annual_ret(p[0]) for p in pts])
        fw.insert(0, "Point", range(1, len(fw) + 1))
        name = f"Frontier S{s['stage']}"
        fw.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for row in ws.iter_rows(min_row=2, min_col=2):
            for c in row:
                header = ws.cell(row=1, column=c.column).value
                c.number_format = NUM if header == "Sharpe (annual)" else PCT
        ws.freeze_panes = "E2"
        autosize(ws, 9, 7)

    # Inputs and notes
    inputs = info.copy()
    inputs["Expected return (per period)"] = expected_returns
    inputs["Expected return (annual)"] = annual_ret(expected_returns)
    inputs["Volatility used (per period)"] = np.sqrt(np.diag(cov))
    inputs["Volatility used (annual)"] = annual_vol(np.sqrt(np.diag(cov)))
    inputs.rename_axis("ticker").to_excel(writer, sheet_name="Inputs")
    ws = writer.sheets["Inputs"]
    for row in ws.iter_rows(min_row=2, min_col=4):
        for c in row:
            c.number_format = PCT
    autosize(ws, 16)
    ws.column_dimensions["B"].width = 24

    notes = pd.DataFrame({"Setting": [
        "Covariance matrix", "Expected returns", "Risk-free rate", "Periods per year",
        "Stage 1 constraint", "Later stages", "Stop rule", "Stages run",
    ], "Value": [
        COV_SOURCE,
        f"Mean of {STAGGER} time-weighted returns ({len(stock_returns)} periods)",
        f"{RISK_FREE_RATE:.0%}",
        f"{PERIODS_PER_YEAR:.2f} ({period_days:.0f}-day periods); annual return = per-period x "
        f"{PERIODS_PER_YEAR:.2f}, annual volatility = per-period x sqrt({PERIODS_PER_YEAR:.2f})",
        "Minimum holding 0% for every stock (long-only), weights sum to 100%",
        f"Cap at {MAX_HOLDING:.0%} for holdings above {MAX_HOLDING:.0%}; floor at "
        f"{MIN_HOLDING:.1%} for holdings below {MIN_HOLDING:.1%}; earlier constraints kept",
        f"All max-SR holdings between {MIN_HOLDING:.1%} and {MAX_HOLDING:.0%}",
        len(stages),
    ]})
    notes.to_excel(writer, sheet_name="Notes", index=False)
    writer.sheets["Notes"].column_dimensions["A"].width = 22
    writer.sheets["Notes"].column_dimensions["B"].width = 100

print(f"saved {OUTPUT_FILE.resolve().name} ({len(stages)} stages)")